In [1]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

## **Pydantic**

In [7]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    query: str = Field(description="The query to be answered by the LLM")
    answer: str = Field(description="The answer to the query generated by the LLM")
    total_tokens: int = Field(description="The total number of tokens used in the LLM response (input + output)")


In [ ]:
# This will throw an error
#obj = llm_schema(query="What is the capital of France?", answer="Paris", total_tokens="Total tokens are 100")
obj = llm_schema(query="What is the capital of France?", answer="Paris", total_tokens=100)


In [ ]:
llm_structured = llm.with_structured_output(llm_schema)
response = llm_structured.invoke("What is the capital of France?")
print(response)

query='What is the capital of France?' answer='Paris.' total_tokens=12


12

## Notes: Chains, Prompt Templates, Pydantic Parsing & Messages

### 1. Messages — The Communication Protocol

Every interaction with a chat model is a sequence of **typed messages**. The model doesn't see a flat string — it sees a structured conversation with roles.

#### Message Types

| Message Type | Role | Purpose | Who creates it |
|---|---|---|---|
| `SystemMessage` | `system` | Sets the model's persona, constraints, and instructions. Processed before all other messages. | Developer |
| `HumanMessage` | `user` | The end-user's input or question. | User / Application |
| `AIMessage` | `assistant` | The model's response. May contain text, tool calls, or both. | LLM |
| `ToolMessage` | `tool` | Result of executing a tool call. Linked to a specific `AIMessage` via `tool_call_id`. | Agent runtime |

#### Why Roles Matter

The model was **fine-tuned to behave differently** based on message roles:

- **System** messages are treated as persistent instructions — the model weighs them heavily and rarely contradicts them.
- **Human** messages are treated as requests to fulfill.
- **AI** messages in the history establish the model's prior behavior (few-shot via example conversations).
- **Tool** messages close the loop on tool calls — without them, the model doesn't know what the tool returned.

```python
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a financial analyst. Always respond with structured data."),
    HumanMessage(content="Analyze AAPL's Q3 earnings.")
]
response = llm.invoke(messages)
```

Under the hood, LangChain serializes this to the OpenAI API format:

```json
{
  "messages": [
    {"role": "system", "content": "You are a financial analyst..."},
    {"role": "user", "content": "Analyze AAPL's Q3 earnings."}
  ]
}
```

The system message here **constrains the model's behavior** for the entire conversation. Without it, the same question might get a casual prose answer instead of structured data.

---

### 2. Prompt Templates — Reusable, Parameterized Prompts

Prompt templates solve the problem of **hardcoded prompts**. Instead of string-formatting manually, you define a template with variables that get filled at runtime.

#### `ChatPromptTemplate`

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {domain} expert. Respond in {language}."),
    ("human", "{question}")
])

# At runtime, fill in the variables:
formatted = prompt.invoke({
    "domain": "cybersecurity",
    "language": "English",
    "question": "What is a zero-day exploit?"
})
```

This produces a `ChatPromptValue` containing:
```
[
  SystemMessage("You are a cybersecurity expert. Respond in English."),
  HumanMessage("What is a zero-day exploit?")
]
```

#### Why Not Just Use f-strings?

| Approach | Problem |
|---|---|
| `f"You are a {domain} expert"` | Works for simple cases, but doesn't produce typed `Message` objects. No validation, no role assignment, no composability with chains. |
| `ChatPromptTemplate` | Produces proper `Message` objects, validates that all variables are provided, composes with `|` (pipe) into chains, and integrates with LangChain's tracing/debugging. |

#### `MessagesPlaceholder` — Injecting Dynamic Message Lists

For agents and multi-turn conversations, you often need to inject an entire message history into the middle of a template:

```python
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])
```

At runtime, `chat_history` can be a list of `HumanMessage` / `AIMessage` pairs from prior turns, giving the model conversational context.

---

### 3. Chains — Composing Components with LCEL

A **chain** is a pipeline of components where the output of one step feeds into the next. LangChain Expression Language (LCEL) uses the **pipe operator** (`|`) to compose them.

#### The Simplest Chain

```python
chain = prompt | llm
response = chain.invoke({"domain": "data engineering", "question": "What is a slowly changing dimension?"})
```

What happens step by step:

```
{"domain": "data engineering", "question": "..."}
        │
        ▼
  ┌─────────────┐
  │   Prompt     │  → Fills variables, produces [SystemMessage, HumanMessage]
  │   Template   │
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   LLM       │  → Sends messages to OpenAI API, returns AIMessage
  └──────┬──────┘
         │
         ▼
    AIMessage(content="A slowly changing dimension is...")
```

#### Adding an Output Parser

```python
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
response = chain.invoke({...})  # Returns a plain string instead of AIMessage
```

Now the chain returns a `str` instead of an `AIMessage` object — the `StrOutputParser` extracts the `.content` field.

#### Why Chains Matter

Without chains, you'd write procedural code:

```python
# Without chains — manual plumbing
formatted_messages = prompt.format_messages(domain="data engineering", question="...")
response = llm.invoke(formatted_messages)
text = response.content
parsed = parser.parse(text)
```

With chains, the same logic is declarative and composable:

```python
# With chains — declarative pipeline
chain = prompt | llm | parser
result = chain.invoke({"domain": "data engineering", "question": "..."})
```

Chains also give you **automatic tracing** (LangSmith), **streaming**, **batch execution**, and **async support** for free — none of which you get with manual plumbing.

#### Chain Composition Examples

```python
# Simple Q&A
chain = prompt | llm | StrOutputParser()

# Structured output via Pydantic
chain = prompt | llm.with_structured_output(MySchema)

# With retrieval (RAG)
chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | parser

# Sequential chains — output of one feeds into the next
chain1 = prompt1 | llm | StrOutputParser()
chain2 = prompt2 | llm | StrOutputParser()
full_chain = chain1 | chain2  # chain1's output becomes chain2's input
```

---

### 4. Pydantic & Structured Output — Two Approaches

There are **two distinct ways** to get structured output from an LLM using Pydantic. This notebook demonstrates one; the other uses a parser.

#### Approach A: `with_structured_output()` (This Notebook)

```python
from pydantic import BaseModel, Field

class LLMSchema(BaseModel):
    query: str = Field(description="The query answered by the LLM")
    answer: str = Field(description="The answer to the query")
    total_tokens: int = Field(description="Total tokens used")

llm_structured = llm.with_structured_output(LLMSchema)
response = llm_structured.invoke("What is the capital of France?")
# → LLMSchema(query='What is the capital of France?', answer='Paris.', total_tokens=12)
```

**How it works under the hood:**
1. LangChain converts the Pydantic model into a **JSON Schema**.
2. This schema is sent to OpenAI's API as a `response_format` (structured outputs) or as a tool schema.
3. The LLM is **constrained at the decoding level** — it can only generate tokens that produce valid JSON matching the schema.
4. The JSON response is parsed back into a Pydantic object.

The model **never sees free-text instructions** about the format. The constraint is enforced by the API itself, making it extremely reliable.

#### Approach B: `PydanticOutputParser` (Text-Based Parsing)

```python
from langchain.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=LLMSchema)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the query.\n{format_instructions}"),
    ("human", "{query}")
])

chain = prompt | llm | parser
response = chain.invoke({
    "query": "What is the capital of France?",
    "format_instructions": parser.get_format_instructions()
})
```

**How it works:**
1. `parser.get_format_instructions()` generates a text prompt like:
   ```
   The output should be formatted as a JSON instance that conforms to the JSON schema below.
   {"properties": {"query": {"type": "string"}, "answer": {"type": "string"}, "total_tokens": {"type": "integer"}}}
   ```
2. This text is injected into the prompt — the model is **asked** (not forced) to output JSON.
3. The model generates free-text JSON in its response.
4. The parser extracts the JSON and validates it against the Pydantic model.

#### Approach A vs Approach B

| Aspect | `with_structured_output()` | `PydanticOutputParser` |
|---|---|---|
| **Reliability** | Very high — schema enforced at API/decoding level | Lower — model may produce malformed JSON |
| **How it works** | JSON Schema sent as `response_format` or tool call | Text instructions injected into prompt |
| **Token cost** | Lower — no format instructions in prompt | Higher — format instructions consume tokens |
| **Error handling** | Rare failures (API guarantees schema compliance) | Needs retry logic for parsing failures |
| **Flexibility** | Tied to providers that support structured output | Works with any LLM that can generate text |
| **Use case** | Production systems, reliable pipelines | Legacy models, custom formats, provider-agnostic code |

**Recommendation**: Use `with_structured_output()` when your provider supports it (OpenAI, Anthropic, Google). Fall back to `PydanticOutputParser` for providers that don't.

---

### 5. What Pydantic Does (Independent of LangChain)

Pydantic is a **data validation library** — it enforces types at runtime, which Python normally doesn't do.

```python
class LLMSchema(BaseModel):
    query: str
    answer: str
    total_tokens: int

# This works:
obj = LLMSchema(query="What is 2+2?", answer="4", total_tokens=100)

# This raises ValidationError — "hello" is not an int:
obj = LLMSchema(query="What is 2+2?", answer="4", total_tokens="hello")
```

In the LLM context, Pydantic serves two purposes:
1. **Schema generation** — `LLMSchema.model_json_schema()` produces the JSON Schema that LangChain sends to the API.
2. **Response validation** — when the LLM returns JSON, Pydantic validates that all fields have correct types before your code touches the data.

The `Field(description=...)` metadata is critical — these descriptions are included in the JSON Schema and tell the LLM **what each field means**, so it can populate them correctly.

---

### 6. Putting It All Together — The Full Pipeline

```
User Input (dict)
      │
      ▼
┌──────────────────┐
│  Prompt Template  │  → Injects variables into typed Messages
│  (parameterized)  │     (SystemMessage + HumanMessage)
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  LLM             │  → Sends messages to API, returns response
│  (ChatOpenAI)    │     constrained by structured output schema
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│  Output Parser   │  → Validates & converts response to
│  (Pydantic)      │     a typed Python object
└────────┬─────────┘
         │
         ▼
   Pydantic Object (validated, typed)
```

Expressed as a chain:

```python
chain = prompt | llm.with_structured_output(LLMSchema)
# or
chain = prompt | llm | PydanticOutputParser(pydantic_object=LLMSchema)
```

### Why This Matters for Agents

Every concept here feeds into the agent pattern from notebooks 1 and 2:

- **Messages** → agents maintain a message list as state (Human → AI → Tool → AI)
- **Prompt templates** → the agent's system prompt defines its persona and instructions
- **Chains** → the agent's internal logic is a chain (prompt → LLM → tool router → tool executor)
- **Structured output** → tool calls are themselves structured output (the model emits JSON with function name + args)

Tool calling (`bind_tools`) is actually a special case of structured output — the model outputs a structured JSON object conforming to the tool's schema, rather than free-form text.

### Sources
- [LangChain LCEL Documentation](https://python.langchain.com/docs/concepts/lcel/)
- [LangChain Prompt Templates](https://python.langchain.com/docs/concepts/prompt_templates/)
- [LangChain Structured Output](https://python.langchain.com/docs/concepts/structured_outputs/)
- [LangChain Output Parsers](https://python.langchain.com/docs/concepts/output_parsers/)
- [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)
- [Pydantic Documentation](https://docs.pydantic.dev/latest/)